# 🖐 손금 분할 모델 학습 (운세연구소 · 손금)

Roboflow의 **palmistry 데이터셋**(생명선·두뇌선·감정선, 1,083장, CC BY 4.0)으로 YOLO11 분할 모델을 학습하고,
사이트에서 쓸 **`palm-lines.onnx`** 파일을 만듭니다.

**사용법:** 각 칸 왼쪽의 ▶ 버튼을 **위에서부터 하나씩** 누르세요. 한 칸이 끝나야(▶ 옆에 ✔) 다음 칸으로 넘어가면 됩니다.

| 순서 | 하는 일 |
|---|---|
| 0 | GPU 켜기 |
| 1 | 프로그램 설치 |
| 2 | 데이터 내려받기 (Roboflow API 키 필요) |
| 3 | 학습 (가장 오래 걸림) |
| 4 | 결과 확인 |
| 5 | ONNX 파일 만들고 내려받기 |

## 0. GPU 켜기
메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU → 저장**을 누른 뒤, 아래 칸을 실행하세요.
표가 나오면 GPU가 켜진 것입니다. `command not found`가 나오면 GPU가 꺼져 있는 상태예요.

In [ ]:
!nvidia-smi

## 1. 프로그램 설치 (1~2분)

In [ ]:
!pip install -q ultralytics roboflow

## 2. 데이터 내려받기
1. Roboflow 데이터셋 페이지에서 복사해 둔 코드를 보고, `project.version(숫자)`의 **숫자**를 아래 `VERSION`에 적으세요.
2. 칸을 실행하면 입력창이 나옵니다. 복사한 코드 속 **API 키**(`api_key="..."` 따옴표 안)를 붙여넣고 Enter.

> 🔒 API 키는 입력창으로만 받기 때문에 이 노트북 파일에는 저장되지 않아요. 키를 채팅이나 GitHub에 올리지 마세요.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

VERSION = 4  # ← Roboflow에서 복사한 코드의 project.version(숫자)와 같게 바꾸세요

api_key = getpass("Roboflow API 키를 붙여넣고 Enter: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("nikhil-singh-s-workspace-k0mu1").project("palmistry-zbcbn-9sz9z")
dataset = project.version(VERSION).download("yolov11")
print("데이터 위치:", dataset.location)

데이터 설정 파일(data.yaml)의 경로를 Colab에 맞게 정리합니다. 클래스 이름과 사진 장수가 출력되면 성공이에요.

In [ ]:
import os, yaml

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data["path"] = dataset.location
for split, folder in [("train", "train"), ("val", "valid"), ("test", "test")]:
    if os.path.isdir(os.path.join(dataset.location, folder, "images")):
        data[split] = f"{folder}/images"
    else:
        data.pop(split, None)
if "val" not in data:          # 검증용 폴더가 없으면 학습용으로 대신 확인
    data["val"] = data["train"]

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, allow_unicode=True)

print("클래스:", data["names"])
for folder in ["train", "valid", "test"]:
    p = os.path.join(dataset.location, folder, "images")
    if os.path.isdir(p):
        print(f"{folder}: {len(os.listdir(p))}장")

## 3. 학습
- 무료 GPU 기준으로 **수십 분 이상** 걸릴 수 있어요. 진행 중에는 이 탭을 닫지 마세요. (런타임이 끊기면 처음부터 다시 해야 해요)
- 줄마다 `Epoch 1/80`처럼 진행 상황이 나옵니다. 성능이 더 이상 좋아지지 않으면 80번 전에 자동으로 멈춰요.
- `yolo11n-seg`는 가장 가벼운 분할 모델이라 브라우저에서도 빠르게 돌아갑니다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")
model.train(
    data=yaml_path,
    epochs=80,       # 최대 학습 횟수
    imgsz=640,       # 사이트 코드(config.ts의 INPUT_SIZE)와 같아야 함
    batch=16,
    patience=20,     # 20번 연속 좋아지지 않으면 멈춤
    project="palm",
    name="lines",
    exist_ok=True,
)

## 4. 결과 확인
`마스크 mAP50`은 0~1 사이 점수로, **1에 가까울수록** 손금을 정확하게 찾는다는 뜻이에요.

In [ ]:
import glob
from ultralytics import YOLO

# ultralytics 버전에 따라 결과 폴더가 palm/lines 또는 runs/segment/palm/lines 등으로 달라서, 어디 있든 찾아서 사용
found = sorted(glob.glob("/content/**/palm/lines/weights/best.pt", recursive=True))
assert found, "best.pt를 찾지 못했어요. 3번 학습 칸이 끝까지(✔) 실행됐는지 확인하세요."
print("찾은 파일:", found[0])

best = YOLO(found[0])
metrics = best.val(data=yaml_path)
print("마스크 mAP50:", round(float(metrics.seg.map50), 3))

(선택) 내 손바닥 사진을 올려서 손금을 잘 찾는지 직접 확인해 보세요.

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
for name in uploaded:
    r = best.predict(name, conf=0.25, verbose=False)[0]
    display(Image.fromarray(r.plot()[..., ::-1]))  # 색 순서 BGR → RGB

## 5. ONNX 파일 만들고 내려받기
실행하면 **`palm-lines.onnx`**와 **`palm-lines.json`** 두 파일이 내려받아집니다.
(브라우저가 '여러 파일 다운로드'를 물어보면 허용)

받은 두 파일은 **`fortune-lab/public/models/palm/`** 폴더에 넣으면 사이트에서 바로 사용돼요.

In [ ]:
import json, os, shutil

onnx_path = best.export(format="onnx", imgsz=640, simplify=True, dynamic=False)
shutil.copy(onnx_path, "palm-lines.onnx")

names = [best.names[i] for i in range(len(best.names))]
with open("palm-lines.json", "w") as f:
    json.dump({"names": names, "imgsz": 640}, f, ensure_ascii=False)

print("클래스 순서:", names)
print("파일 크기:", round(os.path.getsize("palm-lines.onnx") / 1e6, 1), "MB")

files.download("palm-lines.onnx")
files.download("palm-lines.json")

---
**데이터 출처** — palmistry Dataset, Nikhil Singh's Workspace, Roboflow Universe (CC BY 4.0)
https://universe.roboflow.com/nikhil-singh-s-workspace-k0mu1/palmistry-zbcbn-9sz9z
(사이트 손금 결과 화면 아래에도 출처를 표기해 두었습니다.)